In [1]:
import numpy as np
from scipy.spatial.transform import Rotation
import plotly.graph_objects as go
import opengate as gate
import qmirt_utility as qmirt
import pandas as pd
from pathlib import Path
import pyvista as pv
import os
# 1. 必须在导入 pyvista 之前，强行设置环境变量使用 CPU 软件渲染
os.environ["LIBGL_ALWAYS_SOFTWARE"] = "1"

pv.OFF_SCREEN = True

# 3. 尝试启动虚拟显示（这会解决底层 vtkXOpenGLRenderWindow 的报错）
try:
    pv.start_xvfb()
except Exception:
    pass

In [2]:


# Generate a grid of 3D Boxes with center at the origin
def generate_grid_points(grid: np.ndarray, spacing: np.ndarray) -> np.ndarray:
    x_grid, y_grid, z_grid = [
        np.arange(n) * s - (n-1)*s * 0.5
        for n, s, in zip(grid, spacing)
    ]
    return np.stack(np.meshgrid(x_grid, y_grid, z_grid, indexing='xy'), axis=-1)

test_grid_points = generate_grid_points(np.array([3, 3, 1]), np.array([1, 1, 1]))

def generate_box_at_points(points: np.ndarray, box_size: np.ndarray) -> np.ndarray:
    # Create a box centered at the origin
    half_size = box_size / 2
    box_corners = np.array([
        [-half_size[0], -half_size[1], -half_size[2]],
        [ half_size[0], -half_size[1], -half_size[2]],
        [ half_size[0],  half_size[1], -half_size[2]],
        [-half_size[0],  half_size[1], -half_size[2]],
        [-half_size[0], -half_size[1],  half_size[2]],
        [ half_size[0], -half_size[1],  half_size[2]],
        [ half_size[0],  half_size[1],  half_size[2]],
        [-half_size[0],  half_size[1],  half_size[2]],
    ])
    
    # Translate the box to each point in the grid
    boxes_vertices = points[:, None,:] + box_corners

    # Get boxes edges
    return boxes_vertices


In [3]:
n_heads = 20
n_rows = 4
radius_mm = 200.0  # distance from the origin to each head
angles_deg = np.linspace(0, 360, n_heads//n_rows, endpoint=False)
# Repeat the angles 4 times to get 80 heads
# angles_deg = np.tile(angles_deg, 2)
angles_deg = np.tile(angles_deg.reshape(-1, 1), (n_rows, 1)).flatten()
print(f"Angles shape: {angles_deg.shape}")

n_pixels = np.array([2, 2, 1])
monolithic_crystal_size = np.array([50.0, 50.0, 10.0])
row_height_mm = 30.0
monolithic_crystal_translation_z = (
    np.tile(np.arange(n_rows).reshape(-1, 1), (1, n_heads // n_rows)) * row_height_mm
    - (n_rows - 1) * row_height_mm * 0.5
).flatten()
print(f"Monolithic crystal translation Z shape: {monolithic_crystal_translation_z.shape}")
# print(f"Monolithic crystal translation Z: {monolithic_crystal_translation_z}")

monolithic_crystal_translation = np.column_stack(
    (
        radius_mm * np.cos(np.radians(angles_deg)),
        radius_mm * np.sin(np.radians(angles_deg)),
        monolithic_crystal_translation_z,
    )
)

print(f"Monolithic crystal translation shape: {monolithic_crystal_translation.shape}")
print(f"Monolithic crystal translation: {monolithic_crystal_translation}")

pixel_box_size = monolithic_crystal_size / n_pixels

local_array_translations = gate.geometry.utility.get_grid_repetition(
        size=n_pixels, spacing=pixel_box_size
    )

global_pixel_translations = []
for i in range(n_heads):
        global_pixel_translations.extend(
            [
                local_translation + monolithic_crystal_translation[i]
                for local_translation in local_array_translations
            ]
        )
print(
    f"Global pixel translations shape: {np.array(global_pixel_translations).shape}"
)

boxes = generate_box_at_points(np.array(global_pixel_translations), pixel_box_size)
# # # Plot the boxes
# box_edges_indices = np.array([
#     [0, 1], [1, 2], [2, 3], [3, 0],  # Bottom face
#     [4, 5], [5, 6], [6, 7], [7, 4],  # Top face
#     [0, 4], [1, 5], [2, 6], [3, 7]   # Vertical edges
# ])
# print(f"Boxes shape: {boxes.shape}")
# # Rotate the boxes
# rotation = Rotation.from_euler('z', 45, degrees=True)
# rotated_boxes = rotation.apply(boxes.reshape(-1, 3)).reshape(boxes.shape)
# print(rotated_boxes.shape)

Angles shape: (20,)
Monolithic crystal translation Z shape: (20,)
Monolithic crystal translation shape: (20, 3)
Monolithic crystal translation: [[ 200.            0.          -45.        ]
 [  61.80339887  190.21130326  -45.        ]
 [-161.80339887  117.55705046  -45.        ]
 [-161.80339887 -117.55705046  -45.        ]
 [  61.80339887 -190.21130326  -45.        ]
 [ 200.            0.          -15.        ]
 [  61.80339887  190.21130326  -15.        ]
 [-161.80339887  117.55705046  -15.        ]
 [-161.80339887 -117.55705046  -15.        ]
 [  61.80339887 -190.21130326  -15.        ]
 [ 200.            0.           15.        ]
 [  61.80339887  190.21130326   15.        ]
 [-161.80339887  117.55705046   15.        ]
 [-161.80339887 -117.55705046   15.        ]
 [  61.80339887 -190.21130326   15.        ]
 [ 200.            0.           45.        ]
 [  61.80339887  190.21130326   45.        ]
 [-161.80339887  117.55705046   45.        ]
 [-161.80339887 -117.55705046   45.        ]
 

In [ ]:

fig = go.Figure()

box_faces_ijk = np.array([
    [0, 1, 2], [0, 2, 3],  # Bottom face
    [4, 5, 6], [4, 6, 7],  # Top face
    [0, 1, 5], [1, 5, 6],   # Vertical faces
    [1, 2, 6], [2, 6, 7],
    [2, 3, 7], [3, 7, 4],
    [3, 0, 4], [0, 4, 5]
])
print(f"Box faces shape: {box_faces_ijk.shape}")
go_mesh_data = []
for box in boxes:  # Example: only the first box
    #Construct mesh3d data from the vertices
    go_mesh_data.append(
        go.Mesh3d(
            x=box[:, 0],
            y=box[:, 1],
            z=box[:, 2],
            i=box_faces_ijk[:, 0],
            j=box_faces_ijk[:, 1],
            k=box_faces_ijk[:, 2],
            opacity=0.5,
            color='blue'
        )
    )
fig.add_traces(go_mesh_data)
fig.update_layout(scene=dict(
    xaxis_title='X Axis',
    yaxis_title='Y Axis',
    zaxis_title='Z Axis',
    #
    aspectmode='data',
    aspectratio=dict(x=1, y=1, z=1)
))
fig.show()

In [4]:
style_sheet_path = "wrl_stylesheet_simple.json"
fig = qmirt.plot.plot_wrl_file(
    "../test_repeat_box_geometry.wrl",
    style_sheet=style_sheet_path,
    exclude_patterns=["world:*"],
    # exclude_patterns=["crystal:*", "world:*"],
    exclude_mode="replace",
)
fig.show()

In [4]:
def get_dc_spect_geometry_config(xlsx_path: Path):

    n_heads = 80
    collimator_hole_size_mm = 2.3  # unit is mm
    collimator_wall_thickness_mm = 2.0  # unit is mm
    collimator_guide_length_mm = 3.0
    detector_crystal_size_mm = [50.0, 50.0, 10.0]  # unit is mm

    df_coords = pd.read_excel(
        xlsx_path, sheet_name="Coordinates"
    )  # read the "Coordinates" sheet
    df_coords.columns = df_coords.iloc[0]
    df_coords = df_coords[1:]  # remove the first row which is now the header
    df_coords = df_coords.reset_index(
        drop=True
    )  # reset the index after removing the first row
    df_coords = df_coords.apply(
        pd.to_numeric, errors="coerce"
    )  # convert all columns to numeric, coercing errors to NaN
    df_coords.columns.name = "Coordinates Sheet"

    collimator_body_length_mm_np = df_coords["length of collimator"].values
    collimator_hole_coords_mm = df_coords[
        [
            "x coordinate value at center of hole",
            "y coordinate value at center of hole",
            "z coordinate value at center of hole",
        ]
    ].values

    # Fail early if the spreadsheet contains non-numeric values in required fields.
    if (
        np.isnan(collimator_body_length_mm_np).any()
        or np.isnan(collimator_hole_coords_mm).any()
    ):
        raise ValueError(
            "Invalid geometry spreadsheet values detected (NaN) in required columns. "
            "Please check the Coordinates sheet numeric fields."
        )

    if not isinstance(collimator_body_length_mm_np, np.ndarray):
        collimator_body_length_mm_np = np.asarray(collimator_body_length_mm_np)

    if not isinstance(collimator_hole_coords_mm, np.ndarray):
        collimator_hole_coords_mm_np = np.asarray(collimator_hole_coords_mm)
    else:
        collimator_hole_coords_mm_np = collimator_hole_coords_mm

    hole_fov_center_distance_mm_np = np.linalg.norm(
        collimator_hole_coords_mm_np, axis=1
    )
    azmuthal_angle_deg = (
        np.arctan2(
            collimator_hole_coords_mm_np[:, 1], collimator_hole_coords_mm_np[:, 0]
        )
        * 180
        / np.pi
    )
    hole_fov_center_dist_xy_mm_np = np.linalg.norm(
        collimator_hole_coords_mm_np[:, :2], axis=1
    )
    polar_angle_deg = (
        np.arctan2(collimator_hole_coords_mm_np[:, 2], hole_fov_center_dist_xy_mm_np)
        * 180
        / np.pi
    )
    collimator_body_center_dist_mm_np = (
        hole_fov_center_distance_mm_np + collimator_body_length_mm_np * 0.5
    )
    collimator_body_translation_mm = collimator_body_center_dist_mm_np.reshape(
        -1, 1
    ) * np.column_stack(
        (
            np.cos(np.radians(polar_angle_deg))
            * np.cos(np.radians(azmuthal_angle_deg)),
            np.cos(np.radians(polar_angle_deg))
            * np.sin(np.radians(azmuthal_angle_deg)),
            np.sin(np.radians(polar_angle_deg)),
        )
    )
    detector_crystal_center_dist_mm_np = (
        hole_fov_center_distance_mm_np
        + collimator_body_length_mm_np
        + detector_crystal_size_mm[2] * 0.5
    )
    detector_crystal_translation_mm = detector_crystal_center_dist_mm_np.reshape(
        -1, 1
    ) * np.column_stack(
        (
            np.cos(np.radians(polar_angle_deg))
            * np.cos(np.radians(azmuthal_angle_deg)),
            np.cos(np.radians(polar_angle_deg))
            * np.sin(np.radians(azmuthal_angle_deg)),
            np.sin(np.radians(polar_angle_deg)),
        )
    )

    collimator_wall_thickness_mm_np = np.full((n_heads,), collimator_wall_thickness_mm)
    collimator_body_inner_top_mm_np = np.full((n_heads,), detector_crystal_size_mm[0])
    collimator_body_inner_bottom_mm_np = np.full((n_heads,), collimator_hole_size_mm)
    collimator_body_outer_top_mm_np = (
        collimator_body_inner_top_mm_np + collimator_wall_thickness_mm_np * 2
    )
    collimator_body_outer_bottom_mm_np = (
        collimator_body_inner_bottom_mm_np + collimator_wall_thickness_mm_np * 2
    )

    collimator_guide_exit_angle_rad = np.arctan2(
        (collimator_body_inner_top_mm_np + collimator_body_inner_bottom_mm_np) * 0.5,
        collimator_body_length_mm_np,
    )

    collimator_guide_length_mm_np = np.full((n_heads,), collimator_guide_length_mm)
    collimator_guide_distance_mm_np = (
        hole_fov_center_distance_mm_np - collimator_guide_length_mm_np
    )
    collimator_guide_translation_mm = collimator_guide_distance_mm_np.reshape(
        -1, 1
    ) * np.column_stack(
        (
            np.cos(np.radians(polar_angle_deg))
            * np.cos(np.radians(azmuthal_angle_deg)),
            np.cos(np.radians(polar_angle_deg))
            * np.sin(np.radians(azmuthal_angle_deg)),
            np.sin(np.radians(polar_angle_deg)),
        )
    )

    collimator_guide_inner_top_mm_np = np.full((n_heads,), collimator_hole_size_mm)
    collimator_guide_outer_top_mm_np = (
        collimator_guide_inner_top_mm_np + collimator_wall_thickness_mm_np * 2
    )
    collimator_guide_inner_bottom_mm_np = (
        collimator_guide_inner_top_mm_np
        + np.tan(collimator_guide_exit_angle_rad) * collimator_guide_length_mm_np * 2
    )
    collimator_guide_outer_bottom_mm_np = (
        collimator_guide_inner_bottom_mm_np + collimator_wall_thickness_mm_np * 2
    )

    return {
        "collimator_body_length_mm_np": collimator_body_length_mm_np,
        "collimator_hole_coords_mm_np": collimator_hole_coords_mm_np,
        "collimator_body_translation_mm": collimator_body_translation_mm,
        "collimator_body_inner_top_mm_np": collimator_body_inner_top_mm_np,
        "collimator_body_inner_bottom_mm_np": collimator_body_inner_bottom_mm_np,
        "collimator_body_outer_top_mm_np": collimator_body_outer_top_mm_np,
        "collimator_body_outer_bottom_mm_np": collimator_body_outer_bottom_mm_np,
        "collimator_guide_length_mm_np": collimator_guide_length_mm_np,
        "collimator_guide_translation_mm": collimator_guide_translation_mm,
        "collimator_guide_inner_top_mm_np": collimator_guide_inner_top_mm_np,
        "collimator_guide_outer_top_mm_np": collimator_guide_outer_top_mm_np,
        "collimator_guide_inner_bottom_mm_np": collimator_guide_inner_bottom_mm_np,
        "collimator_guide_outer_bottom_mm_np": collimator_guide_outer_bottom_mm_np,
        "detector_crystal_size_mm": detector_crystal_size_mm,
        "detector_crystal_translation_mm": detector_crystal_translation_mm,
        "azmuthal_angle_deg": azmuthal_angle_deg,
        "polar_angle_deg": polar_angle_deg,
    }



In [9]:

dc_spect_config = get_dc_spect_geometry_config(
    Path("/home/fanghan/Work/RPIL/QMIRT/gate10mc/persistent_data/")
    / "spreadsheet/MDSL.excel80M10RFR.cut-plate.010.150roi.2.30pin.105ellipse.xlsx"
)

azmuthal_angle_deg_raw = dc_spect_config["azmuthal_angle_deg"]
polar_angle_deg_raw = dc_spect_config["polar_angle_deg"]
monolithic_crystal_translation_raw = dc_spect_config["detector_crystal_translation_mm"]

n_pixels = np.array([25, 25, 1])
monolithic_crystal_size = np.array([50.0, 50.0, 10.0])

sampling_interval = 1
azmuthal_angle_deg = azmuthal_angle_deg_raw[::sampling_interval]
polar_angle_deg = polar_angle_deg_raw[::sampling_interval]
monolithic_crystal_translation = monolithic_crystal_translation_raw[::sampling_interval]


pixel_box_size = monolithic_crystal_size / n_pixels

local_array_translations = gate.geometry.utility.get_grid_repetition(
    size=n_pixels, spacing=pixel_box_size
)

global_pixel_translations = []
global_pixel_rotations = []
for i in range(monolithic_crystal_translation.shape[0]):
    # First rotate around x axis by 90 degrees
    rx_0 = Rotation.from_euler("x", -90, degrees=True).as_matrix()
    rz_0 = Rotation.from_euler("z", 90, degrees=True).as_matrix()
    # Then rotate around z axis by the azmuthal angle
    rz_1 = Rotation.from_euler("z", azmuthal_angle_deg[i], degrees=True).as_matrix()

    # # Then rotate around y axis by the polar angle
    rx_1 = Rotation.from_euler("x", -polar_angle_deg[i], degrees=True).as_matrix()
    r = rz_1 @ rz_0 @ rx_1 @ rx_0
    # r = rz_1 @ rz_0
    global_pixel_translations.extend(
        [
            Rotation.from_matrix(r).apply(local_translation) + monolithic_crystal_translation[i]
            for local_translation in local_array_translations
        ]
    )


    global_pixel_rotations.extend([r]*n_pixels.prod())
print(f"Global pixel translations shape: {np.array(global_pixel_translations).shape}")
print(f"Global pixel rotations shape: {np.array(global_pixel_rotations).shape}")

# Create an array of (0,0,0)
zero_list = [np.array([[0, 0, 0]])] * len(global_pixel_translations)
boxes = generate_box_at_points(np.array(zero_list), pixel_box_size)
for i in range(len(boxes)):
    boxes[i] = Rotation.from_matrix(global_pixel_rotations[i]).apply(boxes[i])
boxes= np.array(boxes).reshape(-1, 8, 3) + np.array(global_pixel_translations).reshape(-1, 1, 3)

Global pixel translations shape: (50000, 3)
Global pixel rotations shape: (50000, 3, 3)


In [10]:
vertices = []
box_intrinsic_quad_faces = np.array(
    [
        [4,0, 1, 2, 3],  # Bottom face
        [4,4, 5, 6, 7],  # Top face
        [4,0, 1, 5, 4],  # Front face
        [4,2, 3, 7, 6],  # Back face
        [4,0, 3, 7, 4],  # Left face
        [4,1, 2, 6, 5],  # Right face
    ]
)
faces = []
for i, box in enumerate(boxes):
    vertices.extend(box)
    faces.extend(box_intrinsic_quad_faces + np.stack([0, 8, 8, 8, 8]) * i)

print(f"Vertices shape: {np.array(vertices).shape}")
print(f"Faces shape: {np.array(faces).shape}")

detector_pixels_mesh = pv.PolyData(vertices, np.array(faces))


Vertices shape: (400000, 3)
Faces shape: (300000, 5)


In [12]:


collimator_mesh = pv.read("/home/fanghan/Work/RPIL/QMIRT/gate10mc/persistent_data/stl/MDSL_cloud.009.tungsten.STL")
# 2. 开启无屏渲染模式 (针对远程服务器)
plotter = pv.Plotter()
collimator_mesh.rotate_x(90,inplace=True)  # STL 模型通常需要旋转才能正确显示

# 3. 将模型添加到场景中
# STL 通常是极其密集的三角形网格，建议关闭线框 (show_edges=False) 
# 并开启平滑着色 (smooth_shading=True) 以获得更好的视觉效果
plotter.add_mesh(
    collimator_mesh, 
    color='lightblue',       # 设置模型颜色
    show_edges=False,        # 不显示密集的三角形边框
    smooth_shading=False,     # 开启平滑光照
    specular=0.5             # 增加一点高光让模型更有质感
)
plotter.add_mesh(detector_pixels_mesh, show_edges=True, color='cyan')
# 4. 导出为 HTML，随后在 Jupyter Lab 左侧文件栏双击打开即可
plotter.export_html("my_stl_render.html")
# 5. 关闭 plotter 释放内存
plotter.close()

No matching fbConfigs or visuals found
glx: failed to create drisw screen
No matching fbConfigs or visuals found
glx: failed to create drisw screen
2026-05-12 16:05:02.868 ( 209.060s) [    74D8D4143740]vtkXOpenGLRenderWindow.:732   WARN| vtkXOpenGLRenderWindow (0x60db1a69e8a0): Cannot create GLX context.
2026-05-12 16:05:02.868 ( 209.061s) [    74D8D4143740]vtkOpenGLRenderWindow.c:919   WARN| vtkXOpenGLRenderWindow (0x60db1a69e8a0): Failed to initialize OpenGL functions!
